# Bank Campaign Response Prediction

## Part 1 · Define the pre-call decision

Imagine a campaign team has time for 20 calls. Before dialling, which contacts are most likely to subscribe to a term deposit? Each historical record pairs campaign information with a yes/no response. Our aim is to estimate that response from information the team could have at that moment.

### Load the date-ordered records

UCI distributes several versions of the dataset. We use the full additional file and keep the original row order for a later time-based evaluation. The outer ZIP contains a second ZIP, and the CSV uses semicolons between fields. See the [README](README.md) for download instructions.

In [1]:
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

import pandas as pd

archive = Path("data/bank-marketing.zip")
with ZipFile(archive) as outer:
    with ZipFile(BytesIO(outer.read("bank-additional.zip"))) as inner:
        with inner.open("bank-additional/bank-additional-full.csv") as csv_file:
            campaigns = pd.read_csv(csv_file, sep=";")

print(f"{len(campaigns):,} records | {campaigns.shape[1]} columns")

41,188 records | 21 columns


### Decide what the model may see

| Field | Meaning | Use |
|---|---|---|
| job | Job category recorded for the client | Starting input |
| previous | Number of contacts before the current campaign | Starting input |
| poutcome | Outcome of the previous campaign | Starting input |
| duration | Length of the current call in seconds | Exclude because it is unknown before dialling |
| y | Whether the client subscribed | Answer used for learning and evaluation |

In [2]:
target = "y"
pre_call_features = ["job", "previous", "poutcome"]
assert campaigns.shape == (41188, 21), "Check the dataset variant."
assert campaigns[target].notna().all()
assert set(campaigns[target].unique()) == {"yes", "no"}
assert {target, "duration"}.isdisjoint(pre_call_features)

print("Target labels:", sorted(campaigns[target].unique()))
print(campaigns[pre_call_features].head(3).to_string(index=False))

Target labels: ['no', 'yes']
      job  previous    poutcome
housemaid         0 nonexistent
 services         0 nonexistent
 services         0 nonexistent


In the first row, `previous = 0` and `poutcome = nonexistent` describe a client with no previous campaign contact recorded. A previous failure and no previous campaign are different situations. We will preserve that distinction when preparing the data.

### Why the timing matters

A ten-minute conversation may tell us something about interest, but its length is impossible to enter one second before dialling. Including it could make an offline score look better than the real tool could deliver. We start with an explicit list of permitted inputs so that adding a column later requires a reason.

**Next milestone:** Reserve later records before inspecting response rates or choosing features using outcomes. The app will estimate response among contacted customers; this dataset alone cannot establish how many subscriptions were caused by a call.